# Module 08: 3D Plots and Contours

Every plot so far has mapped data onto two axes. That works well when one variable responds to one other, but many real systems have a quantity that depends on two independent variables simultaneously: a temperature field across a surface, a yield that responds to both reagent ratio and reaction time, or the shape of a polymer chain in space. This module introduces the tools Matplotlib provides for those cases, from three-dimensional scatter and surface plots down to the contour views that let you read a 3D surface as a flat map.

## Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from mpl_toolkits.mplot3d import Axes3D   # registers the '3d' projection; must be imported even if not used directly
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

## 1. Setting up a 3D axis

Matplotlib handles three-dimensional plots through a projection system. When you create a subplot, passing `projection='3d'` tells Matplotlib to use the `Axes3D` class from `mpl_toolkits.mplot3d` instead of the default flat axes. The result is an axes object with the same interface you already know (`set_xlabel`, `set_title`, `legend`) plus additional methods for 3D geometry. Importing `mpl_toolkits.mplot3d` registers the projection so that the string `'3d'` is recognized.

In [ ]:
fig = plt.figure(figsize=(5, 4))

# projection='3d' is the only change needed to get a three-dimensional axes
ax = fig.add_subplot(projection='3d')

# Plot a simple helix to confirm the axes work
t = np.linspace(0, 4 * np.pi, 300)
ax.plot(np.cos(t), np.sin(t), t / (4 * np.pi), color='steelblue', linewidth=1.5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('A 3D axis')

plt.tight_layout()
plt.show()

The 3D axes renders the helix with perspective projection: lines further from the viewer appear shorter. You can click and drag the plot in an interactive Matplotlib window to rotate it, or set the viewing angle in code with `ax.view_init`, which is covered at the end of this module.

## 2. 3D scatter plot: RAFT polymerization optimization

In Bayesian optimization of a polymerization reaction, the goal is to find the combination of reaction conditions that maximizes some target property (yield, dispersity, conversion) while running as few experiments as possible. The process works in rounds: an initial set of experiments covers the space broadly, a surrogate model is fitted to those results, and the model suggests where to experiment next. Plotting the three experimental variables in 3D makes it possible to see directly how the search evolved across rounds and whether the optimizer pushed into previously unexplored regions.

The three axes here are the R/I ratio (initiator loading), the M/R ratio (monomer-to-RAFT-agent ratio), and reaction time. All three span different physical scales, so they are log-normalized to a common 0-to-1 range before plotting.

In [ ]:
# Physical axis bounds for the full Round 2 search space
ri_lo,  ri_hi  = 0.5,  10.0   # R/I ratio
mr_lo,  mr_hi  = 15.0, 36.0   # M/R ratio
time_lo, time_hi = 0.1, 20.0  # reaction time (min)

def log_norm(values, lo, hi):
    """Normalize values to [0, 1] on a log scale."""
    return (np.log(values) - np.log(lo)) / (np.log(hi) - np.log(lo))

# --- Initial design (9 points): space-filling across the full range ---
rng = np.random.default_rng(seed=7)

ri_init   = np.array([0.6,  1.2,  2.5,  5.0,  9.0,  0.8,  3.5,  7.0,  1.8])
mr_init   = np.array([16.0, 20.0, 32.0, 18.0, 28.0, 35.0, 24.0, 16.5, 30.0])
time_init = np.array([0.2,  1.0,  5.0,  10.0, 18.0, 3.0,  0.5,  7.0,  14.0])

# --- Round 1 Bayesian optimization (10 points): focused on shorter times, mid M/R ---
ri_r1   = np.array([1.0,  2.0,  4.0,  6.0,  8.0,  1.5,  3.0,  5.5,  7.5,  2.8])
mr_r1   = np.array([15.5, 17.0, 19.0, 22.0, 24.5, 16.0, 21.0, 23.0, 18.5, 20.5])
time_r1 = np.array([0.15, 0.4,  1.2,  2.5,  4.5,  0.8,  1.8,  3.2,  0.3,  2.0])

# --- Round 2 Bayesian optimization (9 points): expanded into higher M/R and longer times ---
ri_r2   = np.array([1.2,  3.5,  6.0,  9.5,  2.0,  4.5,  7.0,  1.5,  5.0])
mr_r2   = np.array([26.0, 29.0, 32.0, 34.5, 27.5, 31.0, 35.5, 28.0, 33.0])
time_r2 = np.array([6.0,  9.0,  13.0, 18.5, 7.5,  11.0, 16.0, 8.5,  14.5])

# Normalize all groups to [0, 1] log scale
def norm_group(ri, mr, time):
    return log_norm(ri, ri_lo, ri_hi), log_norm(mr, mr_lo, mr_hi), log_norm(time, time_lo, time_hi)

xi, yi, zi = norm_group(ri_init, mr_init, time_init)
x1, y1, z1 = norm_group(ri_r1,  mr_r1,  time_r1)
x2, y2, z2 = norm_group(ri_r2,  mr_r2,  time_r2)

In [ ]:
# Normalized bounds for the our rounds
r1_box = {
    'ri':   log_norm(np.array([min(ri_r1),   max(ri_r1)]),   ri_lo,   ri_hi),
    'mr':   log_norm(np.array([min(mr_r1),   max(mr_r1)]),   mr_lo,   mr_hi),
    'time': log_norm(np.array([min(time_r1), max(time_r1)]), time_lo, time_hi),
}

r2_box = {
    'ri':   log_norm(np.array([min(ri_r2),   max(ri_r2)]),   ri_lo,   ri_hi),
    'mr':   log_norm(np.array([min(mr_r2),   max(mr_r2)]),   mr_lo,   mr_hi),
    'time': log_norm(np.array([min(time_r2), max(time_r2)]), time_lo, time_hi),
}

def draw_wireframe_box(ax, xlim, ylim, zlim, color='dimgray', lw=0.8, ls='--'):
    """Draw a rectangular box in 3D using 12 line segments."""
    x0, x1 = xlim
    y0, y1 = ylim
    z0, z1 = zlim
    # Four edges along x
    for y in [y0, y1]:
        for z in [z0, z1]:
            ax.plot3D([x0, x1], [y, y], [z, z], color=color, lw=lw, ls=ls)
    # Four edges along y
    for x in [x0, x1]:
        for z in [z0, z1]:
            ax.plot3D([x, x], [y0, y1], [z, z], color=color, lw=lw, ls=ls)
    # Four edges along z
    for x in [x0, x1]:
        for y in [y0, y1]:
            ax.plot3D([x, x], [y, y], [z0, z1], color=color, lw=lw, ls=ls)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection='3d')

# Earlier rounds at reduced alpha so the progression is readable
ax.scatter(xi, yi, zi, color='gray',    marker='o', s=45, alpha=0.35, label='Initial design')
ax.scatter(x1, y1, z1, color='firebrick', marker='^', s=55, alpha=0.35, label='Round 1 (BO)')
ax.scatter(x2, y2, z2, color='steelblue', marker='x', s=65, alpha=1.0,  label='Round 2 (BO)', linewidths=1.5)

# Wireframe box around the Round 1 search bounds
draw_wireframe_box(ax, xlim=r1_box['ri'], ylim=r1_box['mr'], zlim=r1_box['time'], color='firebrick', lw=0.9, ls='--')
draw_wireframe_box(ax, xlim=r2_box['ri'], ylim=r2_box['mr'], zlim=r2_box['time'], color='steelblue', lw=0.9, ls='--')

ax.set_xlabel('R/I ratio (log norm.)', labelpad=8)
ax.set_ylabel('M/R ratio (log norm.)', labelpad=8)
ax.set_zlabel('Time (log norm.)', labelpad=8)
ax.set_title('RAFT optimization: experimental rounds')
ax.legend(loc='upper left', fontsize=8, framealpha=0.7)

# Set axis limits to the full normalized space
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_zlim(0, 1)

ax.view_init(elev=22, azim=225)
plt.tight_layout()
plt.show()

The dashed red box marks the region explored during Round 1. The Round 2 points (blue crosses, full opacity) lie clearly outside that box in the M/R and time directions, showing that the surrogate model identified more promising territory at higher monomer ratios and longer reaction times. Log-normalizing all three axes before plotting puts them on the same 0-to-1 scale, which prevents a variable with a wide physical range from dominating the visual spread.

## 3. 3D line plot: polymer chain random walk

The freely jointed chain model describes a polymer backbone as a sequence of bonds with fixed length but random orientation. Generating a 3D random walk and plotting it as a continuous line gives an immediate visual sense of the chain's end-to-end distance and the spatial volume it occupies. In 3D, `ax.plot()` works exactly as in 2D but takes a third positional argument for the z coordinates.

In [ ]:
rng_chain = np.random.default_rng(seed=19)
n_bonds = 500
bond_length = 1.0

# Random unit vectors: sample uniformly on the sphere using Gaussian trick
raw_steps = rng_chain.standard_normal((n_bonds, 3))
norms = np.linalg.norm(raw_steps, axis=1, keepdims=True)
unit_steps = raw_steps / norms * bond_length

# Cumulative sum gives the position of each monomer along the chain
chain_xyz = np.cumsum(unit_steps, axis=0)
chain_x, chain_y, chain_z = chain_xyz[:, 0], chain_xyz[:, 1], chain_xyz[:, 2]

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection='3d')

ax.plot(chain_x, chain_y, chain_z, color='steelblue', linewidth=0.7, alpha=0.85)

# Mark chain start and end
ax.scatter(*chain_xyz[0],  color='green',     s=60, zorder=5, label='Chain start')
ax.scatter(*chain_xyz[-1], color='firebrick', s=60, zorder=5, label='Chain end')

# Draw the end-to-end vector
ax.plot3D(
    [chain_x[0], chain_x[-1]],
    [chain_y[0], chain_y[-1]],
    [chain_z[0], chain_z[-1]],
    color='black', linewidth=1.2, linestyle='--', label='End-to-end vector'
)

ax.set_xlabel('x (bond lengths)')
ax.set_ylabel('y (bond lengths)')
ax.set_zlabel('z (bond lengths)')
ax.set_title(f'Freely jointed chain ({n_bonds} bonds)')
ax.legend(fontsize=8, framealpha=0.7)

plt.tight_layout()
plt.show()

The end-to-end vector (dashed line) is much shorter than the total contour length of the chain, which is the defining feature of a random coil. Try increasing `n_bonds` to 2000 and re-running: the chain fills a larger volume but the ratio of end-to-end distance to contour length keeps shrinking, consistent with the $R_{ee} \propto \sqrt{N}$ scaling of the freely jointed chain model.

## 4. Surface plot: injection molding temperature field

In plastic injection molding, hot polymer melt is pushed through a tubular runner before entering the mold cavity. As the melt flows, it loses heat through the runner walls, creating a temperature field that varies along two spatial dimensions: axial position (how far down the tube) and radial position (how close to the cold wall). At any instant in time, the melt near the center of the tube is hotter than the melt near the wall, and the whole cross-section cools as you move further from the injection gate.

This is a case where a 3D surface plot genuinely earns its keep. The two spatial coordinates (axial position and radial position) form a natural 2D domain that maps onto the x and y axes. The z-axis represents fill time, giving the surface physical depth. Temperature, the fourth variable, is encoded as color through the `facecolors` argument. Without 3D, you would need either a sequence of 2D snapshots or an animation to show how the spatial temperature profile evolves during filling. Here, the entire story sits in one figure.

In [ ]:
# --- Geometry and physical parameters ---
z_pos = np.linspace(0, 200, 80)      # axial position along the runner (mm)
r_pos = np.linspace(0, 5, 60)        # radial position from center to wall (mm)
Z_grid, R_grid = np.meshgrid(z_pos, r_pos)

T_inject = 240.0      # injection temperature at gate center (deg C)
T_wall   = 60.0       # runner wall temperature (deg C)
z_decay  = 150.0      # axial decay length (mm)
R_runner = 5.0        # runner radius (mm)

# Temperature model: parabolic radial profile (hottest at center, coolest at wall)
# combined with exponential axial decay from the gate
radial_profile = 1.0 - (R_grid / R_runner) ** 2        # 1 at center, 0 at wall
axial_decay    = np.exp(-Z_grid / z_decay)              # 1 at gate, decays downstream

temp_field = T_wall + (T_inject - T_wall) * radial_profile * axial_decay

In [ ]:
from matplotlib import cm
from matplotlib.colors import Normalize

# We want the z-axis to show fill time, so we pick a single snapshot.
# The temperature field at t = 0 s is the initial spatial distribution.
# For the surface plot, z is a constant (one time slice) and color = temperature.
t_snapshot = 0.0   # seconds into the fill

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(projection='3d')

# Map temperature values to colors using RdBu_r
norm = Normalize(vmin=T_wall, vmax=T_inject)
colors = cm.RdBu_r(norm(temp_field))

# Plot the spatial temperature field as a surface at a fixed fill time
# Z_grid and R_grid are the two spatial axes; the surface sits at z = t_snapshot
time_plane = np.full_like(temp_field, t_snapshot)

surf = ax.plot_surface(
    Z_grid, R_grid, time_plane,
    facecolors=colors,
    linewidth=0,
    antialiased=True,
    shade=False
)

# Add a ScalarMappable for the colorbar (plot_surface with facecolors has no mappable)
mappable = cm.ScalarMappable(cmap='RdBu_r', norm=norm)
mappable.set_array([])
cbar = fig.colorbar(mappable, ax=ax, shrink=0.5, pad=0.1)
cbar.set_label('Temperature (°C)')

ax.set_xlabel('Axial position (mm)', labelpad=8)
ax.set_ylabel('Radial position (mm)', labelpad=8)
ax.set_zlabel('Fill time (s)', labelpad=8)
ax.set_title('Runner temperature field at t = 0 s')

ax.view_init(elev=28, azim=225)
plt.tight_layout()
plt.show()

The single-snapshot surface above shows how temperature varies across the two spatial dimensions at one moment. The red region near the center at the gate end is the hottest melt; the blue edges near the wall and far downstream are the coolest. The color does the work that a z-axis would normally do in a simpler 3D plot, and the z-axis itself is free to represent fill time.

The next cell stacks several of these snapshots at different fill times to build the full 3D picture. Each slice is a spatial temperature map at a particular instant, and the vertical spacing between slices shows the progression of cooling.

In [ ]:
# --- Stack several time slices to show the full 3D structure ---
time_slices = [0.0, 1.0, 2.0, 3.0]   # seconds
t_cool = 15.0                           # characteristic cooling time (s)

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(projection='3d')

norm = Normalize(vmin=T_wall, vmax=T_inject)

for t_val in time_slices:
    # Apply time-dependent cooling factor to the spatial field
    cooling = np.exp(-t_val / t_cool)
    temp_at_t = T_wall + (T_inject - T_wall) * radial_profile * axial_decay * cooling

    colors_t = cm.RdBu_r(norm(temp_at_t))
    time_plane = np.full_like(temp_at_t, t_val)

    ax.plot_surface(
        Z_grid, R_grid, time_plane,
        facecolors=colors_t,
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=0.85
    )

# Colorbar
mappable = cm.ScalarMappable(cmap='RdBu_r', norm=norm)
mappable.set_array([])
cbar = fig.colorbar(mappable, ax=ax, shrink=0.5, pad=0.1)
cbar.set_label('Temperature (°C)')

ax.set_xlabel('Axial position (mm)', labelpad=8)
ax.set_ylabel('Radial position (mm)', labelpad=8)
ax.set_zlabel('Fill time (s)', labelpad=8)
ax.set_title('Runner temperature field: spatial slices over time')

ax.view_init(elev=25, azim=225)
plt.tight_layout()
plt.show()

`plot_surface` requires three 2D arrays of matching shape, which is why `meshgrid` is used to expand the 1D axial and radial vectors into grids. The `facecolors` argument accepts an RGBA array that overrides the default height-based coloring, allowing temperature to be mapped to color independently of the z-axis. Each stacked slice is a complete spatial temperature map at one moment in time. Reading upward through the stack, you can see the hot core (red) shrinking and the cold region (blue) expanding as the runner loses heat to the mold walls. This is the payoff of using all three spatial axes plus color: the full spatiotemporal evolution of the temperature field is visible in a single static figure.

## 5. Wireframe plot

A wireframe shows the same surface geometry as `plot_surface` but renders it as a grid of lines rather than filled polygons. This makes the underlying mesh structure visible and is useful when you want to show shape without implying a continuous color gradient, or when two surfaces would overlap and obscure each other if filled.

Since our injection molding surface uses `facecolors` to encode temperature, a wireframe version necessarily loses that color channel. To make the comparison fair, the left panel below shows a single time-slice surface colored by temperature, and the right panel shows the same geometry as a wireframe. The wireframe reveals the grid structure and the curvature of the radial temperature profile but cannot carry the temperature information.

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5),
    subplot_kw={'projection': '3d'}   # apply 3d projection to both subplots at once
)

ax_surf, ax_wire = axes

# --- Left panel: surface colored by temperature ---
norm = Normalize(vmin=T_wall, vmax=T_inject)
colors = cm.RdBu_r(norm(temp_field))

ax_surf.plot_surface(
    Z_grid, R_grid, temp_field,
    facecolors=colors,
    linewidth=0,
    antialiased=True,
    shade=False
)
ax_surf.set_title('Surface (color = temperature)')
ax_surf.set_xlabel('Axial pos. (mm)')
ax_surf.set_ylabel('Radial pos. (mm)')
ax_surf.set_zlabel('Temp (°C)')
ax_surf.view_init(elev=28, azim=225)

# --- Right panel: wireframe of the same data ---
# rstride and cstride control how many grid rows/columns are used for the mesh
ax_wire.plot_wireframe(
    Z_grid, R_grid, temp_field,
    rstride=5, cstride=5,
    color='steelblue',
    linewidth=0.6,
    alpha=0.7
)
ax_wire.set_title('Wireframe')
ax_wire.set_xlabel('Axial pos. (mm)')
ax_wire.set_ylabel('Radial pos. (mm)')
ax_wire.set_zlabel('Temp (°C)')
ax_wire.view_init(elev=28, azim=225)

plt.suptitle('Runner temperature field: surface vs. wireframe', y=1.01)
plt.tight_layout()
plt.show()

`rstride` and `cstride` set how many rows and columns of the underlying mesh grid are skipped between drawn lines. Smaller values give a denser wireframe; larger values show the surface shape with less visual noise. The wireframe makes the curvature of the radial temperature profile easier to follow because the grid lines act as contour guides. Notice how the lines fan out steeply near the wall (high radial position) where the temperature drops fastest. That structure is present in the surface plot too, but the filled color can mask it at certain viewing angles.

## 6. Contour plot: top-down view of the temperature field

A contour plot is a 2D projection of a 3D surface viewed from directly above. Each contour line connects points of equal value, the same principle as elevation contours on a topographic map. `contourf` draws filled regions between contour levels; `contour` overlays the lines themselves. Using both together is common in scientific figures: the fill communicates magnitude through color, and the overlaid lines give precise iso-value boundaries.

For the runner temperature field, the contour plot shows the axial-radial temperature distribution as a flat map. This is the view you would get if you sliced the runner lengthwise and looked straight down at the cross-section.

In [ ]:
fig, ax_cnt = plt.subplots(figsize=(8, 5))

# Filled contours: the color map encodes temperature magnitude
levels = np.linspace(temp_field.min(), temp_field.max(), 20)
cf = ax_cnt.contourf(Z_grid, R_grid, temp_field, levels=levels, cmap='RdBu_r')

# Overlaid contour lines at a coarser set of levels
line_levels = np.linspace(temp_field.min(), temp_field.max(), 8)
cl = ax_cnt.contour(Z_grid, R_grid, temp_field, levels=line_levels, colors='black', linewidths=0.7, alpha=0.6)

# Label the contour lines with their temperature values
ax_cnt.clabel(cl, inline=True, fontsize=7, fmt='%.0f °C')

cbar_cnt = fig.colorbar(cf, ax=ax_cnt)
cbar_cnt.set_label('Temperature (°C)')

ax_cnt.set_xlabel('Axial position along runner (mm)')
ax_cnt.set_ylabel('Radial position (mm)')
ax_cnt.set_title('Runner temperature field: contour map (t = 0 s)')

plt.tight_layout()
plt.show()

The contour lines curve from the lower-left toward the upper-right, reflecting the combined effect of axial cooling (left to right) and radial cooling (bottom to top, i.e. center to wall). Near the gate at the runner center, the isotherms are widely spaced because the temperature is nearly uniform. Near the wall and far downstream, they bunch together as the steep gradient between the hot core and the cold wall compresses into a narrow band. `clabel` places the temperature values inline along the contour lines, which is more readable than a separate legend when there are many levels.

## 7. Cross-sections of the 3D surface

Slicing a 3D surface at fixed values of one variable produces a family of 2D curves that are often easier to read than the full surface. This is a common pattern in characterization data: you measure a response surface, then present individual cross-sections so that trends along one variable can be compared at a glance.

Here we slice the runner temperature field at five axial positions, producing a radial temperature profile at each location. Each panel shows how temperature drops from the center of the tube to the wall at one distance from the gate.

In [ ]:
# Select five axial positions at which to slice the temperature field
z_slice_values = [0, 50, 100, 150, 200]   # mm from gate
slice_colors = ['firebrick', 'darkorange', 'goldenrod', 'steelblue', 'navy']

fig, axes_grid = plt.subplots(2, 3, figsize=(12, 7), sharey=True)
axes_flat = axes_grid.flatten()

for i, (z_val, color) in enumerate(zip(z_slice_values, slice_colors)):
    # Find the index in z_pos closest to the requested slice position
    z_idx = np.argmin(np.abs(z_pos - z_val))

    # Extract the radial temperature profile at this axial position
    temp_slice = temp_field[:, z_idx]

    ax_s = axes_flat[i]
    ax_s.plot(r_pos, temp_slice, color=color, linewidth=2.0)
    ax_s.set_title(f'z = {z_val} mm', fontsize=10)
    ax_s.set_xlabel('Radial position (mm)')
    ax_s.set_ylabel('Temperature (°C)')
    ax_s.set_ylim(50, 250)
    ax_s.grid(True, linestyle=':', linewidth=0.5, alpha=0.6)

# The sixth panel is unused; hide it
axes_flat[-1].set_visible(False)

plt.suptitle('Radial temperature profile at fixed axial positions', y=1.01)
plt.tight_layout()
plt.show()

Each panel shows the radial temperature profile at one axial position. At the gate (z = 0 mm), the center is at the full injection temperature and the wall is cold, producing a steep parabolic profile. Further downstream, axial cooling has reduced the center temperature, so the overall curve flattens. By z = 200 mm, the entire cross-section is approaching the wall temperature and the center-to-wall gradient is small. The `sharey=True` argument keeps all y-axes identical, which is essential here: without it, the profiles at different positions would appear to have similar shapes even though their absolute temperature ranges differ substantially.

## 8. Controlling the viewing angle with `view_init`

`ax.view_init(elev, azim)` sets the camera position for a 3D axes. `elev` is the elevation angle in degrees above the horizontal plane; `azim` is the azimuthal angle in degrees, rotating the view around the vertical axis. The default is `elev=30, azim=45`. A well-chosen viewing angle can reveal structure that is hidden from the default viewpoint, particularly when points cluster along one axis direction.

In [ ]:
view_angles = [
    {'elev': 22,  'azim': 225, 'title': 'elev=22, azim=225 (standard view)'},
    {'elev': 70,  'azim': 180, 'title': 'elev=70, azim=180 (nearly top-down)'},
]

fig, axes_v = plt.subplots(
    1, 2,
    figsize=(12, 5),
    subplot_kw={'projection': '3d'}
)

for ax_v, params in zip(axes_v, view_angles):
    ax_v.scatter(xi, yi, zi, color='gray',      marker='o', s=40, alpha=0.35, label='Initial')
    ax_v.scatter(x1, y1, z1, color='firebrick', marker='^', s=50, alpha=0.35, label='Round 1')
    ax_v.scatter(x2, y2, z2, color='steelblue', marker='x', s=60, alpha=1.0,  label='Round 2', linewidths=1.5)

    draw_wireframe_box(
        ax_v,
        xlim=r1_box['ri'],
        ylim=r1_box['mr'],
        zlim=r1_box['time'],
        color='firebrick', lw=0.8, ls='--'
    )

    ax_v.set_xlabel('R/I (log norm.)', labelpad=6)
    ax_v.set_ylabel('M/R (log norm.)', labelpad=6)
    ax_v.set_zlabel('Time (log norm.)', labelpad=6)
    ax_v.set_xlim(0, 1)
    ax_v.set_ylim(0, 1)
    ax_v.set_zlim(0, 1)
    ax_v.set_title(params['title'], fontsize=9)
    ax_v.legend(loc='upper left', fontsize=7, framealpha=0.6)
    ax_v.view_init(elev=params['elev'], azim=params['azim'])

plt.suptitle('Same data, two viewing angles', y=1.01)
plt.tight_layout()
plt.show()

The left panel shows the standard oblique view where all three axes are visible. The right panel looks nearly straight down from above, collapsing the time axis and showing the R/I versus M/R distribution as if it were a 2D scatter plot. The top-down view makes it clear that Round 2 points (blue crosses) occupy higher M/R values than either previous round, a pattern that is harder to read from the oblique angle. When preparing a 3D figure for a paper or presentation, it is worth testing several angles and choosing the one that most directly shows the feature you are describing.

## 9. Bonus: volumetric rendering with Plotly

The stacked-slice approach in Section 4 is a workaround for a fundamental limitation of Matplotlib: it can only draw surfaces, not volumes. To show how temperature fills the entire 3D domain of axial position, radial position, and fill time without slicing, you need volumetric rendering. Plotly's `go.Volume` trace does this by layering semi-transparent iso-surfaces through a 3D scalar field, letting you see the hot core glowing through the cooler outer regions. The result is an interactive figure you can rotate and zoom in the browser, which is useful for exploratory analysis even if it is not the right format for a printed publication.

In [ ]:
import plotly.graph_objects as go

# ── Physical parameters (same as Section 4) ──────────────────────────
T_inject = 240.0      # injection temperature at gate center (°C)
T_wall   = 60.0       # runner wall temperature (°C)
z_decay  = 150.0      # axial decay length (mm)
R_runner = 5.0        # runner radius (mm)
t_cool   = 4.0        # characteristic cooling time (s)

# ── Build the 3D grid ────────────────────────────────────────────────
nz, nr, nt = 40, 30, 25   # resolution along each axis

z_pos = np.linspace(0, 200, nz)       # axial position (mm)
r_pos = np.linspace(0, 5, nr)         # radial position (mm)
t_fill = np.linspace(0, 5, nt)        # fill time (s)

Z, R, TT = np.meshgrid(z_pos, r_pos, t_fill, indexing='ij')

# ── Temperature field: radial parabolic × axial exponential × cooling ─
radial_profile = 1.0 - (R / R_runner) ** 2
axial_decay    = np.exp(-Z / z_decay)
time_cooling   = np.exp(-TT / t_cool)

temp_vol = T_wall + (T_inject - T_wall) * radial_profile * axial_decay * time_cooling

# ── Plotly volume trace ──────────────────────────────────────────────
# Flatten the grids into 1D arrays as required by go.Volume
fig = go.Figure(data=go.Volume(
    x=Z.flatten(),
    y=R.flatten(),
    z=TT.flatten(),
    value=temp_vol.flatten(),
    isomin=T_wall + 20,          # skip the coldest shell so the hot core is visible
    isomax=T_inject,
    opacity=0.12,                # controls how transparent each voxel layer is
    surface_count=20,            # number of iso-surfaces layered to build the volume
    colorscale='RdBu_r',
    colorbar=dict(title='Temp (°C)'),
    caps=dict(x_show=False, y_show=False, z_show=False),
))

fig.update_layout(
    title='Runner temperature field — volumetric view',
    scene=dict(
        xaxis_title='Axial position (mm)',
        yaxis_title='Radial position (mm)',
        zaxis_title='Fill time (s)',
    ),
    width=800,
    height=600,
    margin=dict(l=10, r=10, t=40, b=10),
)

# ── Save as interactive HTML (open in any browser to rotate/zoom) ─────
fig.write_html('result/bonus_volumetric.html')
print("Saved: bonus_volumetric.html  (interactive, open in browser)")